# U-Net Exercise: Segmenting Rectangles from Circles


In this exercise, you will build and progressively improve a **U-Net for image segmentation**. The coding exercise is to be completed within 60 minutes. The lines of code which are to be developed are clearly marked as #TODO

We will follow a step-by-step approach, starting with a simple segmentation problem and gradually making it more challenging. At each stage, you will evaluate the model and see how the changes affect its ability to segment rectangles.

### Step 1 — Start with a simple dataset

We begin with just **10 simple images** containing circles and rectangles.

Your first task is to train a basic U-Net to identify the **rectangle pixels** while ignoring the circles and background.

You will train the model using **binary cross-entropy (BCE) loss** and examine how the training and test loss change over time.

### Step 2 — Evaluate the segmentation

Loss alone does not tell us exactly how well the predicted segmentation overlaps with the true rectangle.

We therefore introduce two segmentation metrics:

- **Intersection over Union (IoU)** — measures the overlap between the predicted and true rectangle regions.
- **Dice score** — measures how similar the predicted and true regions are.

You will use these metrics alongside BCE loss to understand the model's performance.

### Step 3 — Improve the U-Net

Next, you will modify the U-Net architecture by adding another encoder/decoder level.

The goal is to investigate whether a deeper network can learn more useful representations for the segmentation task.

### Step 4 — Make the dataset more realistic

The original dataset is deliberately simple. Real images are much less predictable.

We will therefore create a more challenging dataset containing:

- Multiple circles
- Multiple rectangles
- Overlapping shapes
- Different brightness levels
- Random image noise
- Rectangles partially overlapping circles

The model now has to learn to identify rectangles even when the visual appearance of the images becomes more ambiguous.

### Step 5 — Train and evaluate on the harder dataset

We will train a new U-Net on the more challenging dataset and again measure:

- **BCE loss**
- **IoU**
- **Dice score**

We will also visualize the model's predictions to see what it has learned.

### Step 6 — Test generalization

Finally, we will introduce an even more difficult test set with more shapes and higher levels of noise.

The important question is:

> **Can the U-Net still identify rectangles when the images are different and more difficult than the examples it was trained on?**

At the end of the exercise, you will compare the different models and datasets and use the results to understand the relationship between **model architecture, training data, evaluation metrics, and generalization**.

The exercise therefore follows the progression:

**Simple dataset → Simple U-Net → IoU & Dice → Improved U-Net → More challenging dataset → New training → Generalization test → Comparison**

## Setup for local users (VS Code, Jupyter)
Google Colab already has all required libraries installed. If you run this notebook locally, the cell below installs them into your current environment. On Colab it does nothing.

In [ ]:
# Installs the required libraries only when NOT running on Google Colab
if 'google.colab' not in str(get_ipython()):
    %pip install numpy matplotlib torch scikit-image

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
!wget https://raw.githubusercontent.com/eth-bmai-hs26/w3-cx-public/main/unet/shapes_data.py
from shapes_data import print_training_data_set, ShapesDataset, IMAGE_SIZE, generate_sample, display_results

torch.manual_seed(0)
np.random.seed(0)

## Training Data Set
Contains a visualization of the images of the training dataset

In [ ]:
train_dataset = ShapesDataset(10)
test_dataset = ShapesDataset(10)
print_training_data_set(train_dataset)

## U-Net architecture

A **U-Net** is a neural network for **image segmentation**: it takes an image and predicts which pixels belong to which class.  

- **Channels**: are like layers showing different features. Grayscale images have 1 channel; color images have 3. The network creates more channels to detect edges, textures, and shapes.  
- **Encoder**: extracts features and reduces image size, like zooming out while remembering important details.  
- **Bottleneck**: the most compressed representation, containing abstract features.  
- **Decoder**: enlarges the image again and merges features from the encoder to recover spatial details.  
- **Output**: a new image/mask highlighting important parts.  

**In short:** U-Net looks at an image, learns patterns, and produces a mask while keeping the original details.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder path
        #TODO: Complete the code below
        self.enc1 = DoubleConv(None, None)  # TODO: first encoder block
        self.enc2 = DoubleConv(None, None)  # TODO: second encoder block

        self.pool = nn.MaxPool2d(None)  # TODO: downsampling layer
        # Bottleneck
        self.bottleneck = DoubleConv(16, 32)  # Bottom of the U, feature expansion: 32 → 64 channels

        # Decoder path
        self.up2 = nn.ConvTranspose2d(32, 16, 2, 2)     # Upsample from 32 → 16 channels, doubles spatial dimensions
        self.dec2 = DoubleConv(32, 16)                  # After concatenation with skip connection: 16+16=32 → 16

        self.up1 = nn.ConvTranspose2d(16, 8, 2, 2)      # Upsample from 16 → 8 channels
        self.dec1 = DoubleConv(16, 8)                   # After concatenation with skip connection: 8+8=16 → 8

        # Final output layer
        self.out = nn.Conv2d(8, 1, 1)   # 1x1 convolution to reduce to desired output channels (e.g., segmentation mask)


    def forward(self, x):
        # Encoder
        #TODO: Apply first double conv
        e1 = None  # TODO
        #TODO: Pool e1 to reduce size, then apply second double conv
        e2 = None  # TODO

        # Bottleneck
        b = self.bottleneck(self.pool(e2))   # Pool again, then double conv at the bottom of the U

        # Decoder
        d2 = self.up2(b)                   # Upsample bottleneck features
        d2 = torch.cat([d2, e2], dim=1)    # Concatenate skip connection from encoder (same spatial size)
        d2 = self.dec2(d2)                 # Process concatenated features with double conv

        d1 = self.up1(d2)                   # Upsample
        d1 = torch.cat([d1, e1], dim=1)     # Concatenate skip connection from first encoder
        d1 = self.dec1(d1)                  # Process with double conv

        # Output
        return self.out(d1)                 # Final 1x1 convolution to produce output

## Training

# U-Net Training Explained

**Inputs:**  
- The network receives an **image**.  

**Ground truth mask:**  
- For training, each image has a **mask** showing the correct labeling of pixels (e.g., 1 for rectangle, 0 for background).  

**Prediction:**  
- The U-Net outputs a **predicted mask** of the same size as the input.  

**Loss calculation:**  
- Compare the predicted mask with the ground truth mask using a **loss function** (in this scenario we use Binary Cross Entropy loss).  
- The loss measures how close the prediction is to the truth.  

**Backpropagation:**  
- The network updates its weights to **minimize the loss**, improving mask predictions over time.  


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

    train_losses = []
    test_losses = []

    for epoch in range(30):
        # ---- Training phase ----
        train_epoch_loss = 0
        model.train()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

            train_epoch_loss += loss.item()

        train_epoch_loss /= len(train_loader)
        train_losses.append(train_epoch_loss)
        print(f"Epoch {epoch+1:02d} | Loss: {train_epoch_loss:.4f}")

        # ---- Testing phase ----
        test_epoch_loss = 0
        model.eval()

        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                loss = criterion(model(x), y)
                test_epoch_loss += loss.item()

        test_epoch_loss /= len(test_loader)
        test_losses.append(test_epoch_loss)
except Exception as e:
    raise NotImplementedError("Something is missing above. Complete all #TODO sections before training.") from e


plt.plot(train_losses, marker="o", label="Train Loss")
plt.plot(test_losses, marker="o", label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")
plt.title("Training loss")
plt.legend()
plt.grid(True)
plt.show()

## Prediction on a new image

In [ ]:
model.eval()
test_img, test_mask = generate_sample()

x = torch.tensor(test_img).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(x))[0, 0].cpu()

display_results(test_img, pred, test_mask)


# #TODO Your Task

Follow the steps to add a third encoder layer and make our U-Net even more accurate. Try the code as is beforehand!

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder path
        self.enc1 = DoubleConv(1, 8)   # First double conv: input 1 channel → 8 channels
        self.enc2 = DoubleConv(8, 16)  # Second double conv: 8 → 16 channels

        #TODO:add a third encoder layer
        self.enc3 = None  # TODO: third encoder block

        self.pool = nn.MaxPool2d(2)    # Max pooling with 2x2 window, reduces spatial dimensions by half

        # Bottleneck
        #self.bottleneck = DoubleConv(16, 32)  # Bottom of the U, feature expansion: 32 → 64 channels
        #TODO: update the bottleneck to match the output of the third encoder
        self.bottleneck = None  # TODO: bottleneck

        # Decoder path
        #TODO: add a third decoder layer
        self.up3 = None  # TODO: third upsampling layer
        self.dec3 = None  # TODO: third decoder block

        self.up2 = nn.ConvTranspose2d(32, 16, 2, 2)  # Upsample from 32 → 16 channels, doubles spatial dimensions
        self.dec2 = DoubleConv(32, 16)              # After concatenation with skip connection: 16+16=32 → 16

        self.up1 = nn.ConvTranspose2d(16, 8, 2, 2)   # Upsample from 16 → 8 channels
        self.dec1 = DoubleConv(16, 8)               # After concatenation with skip connection: 8+8=16 → 8

        # Final output layer
        self.out = nn.Conv2d(8, 1, 1)   # 1x1 convolution to reduce to desired output channels (e.g., segmentation mask)


    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)                # Apply first double conv
        e2 = self.enc2(self.pool(e1))    # Pool e1 to reduce size, then apply second double conv#
        #TODO:add a third encoder layer
        e3 = None  # TODO
        b = self.bottleneck(self.pool(e3))
        # Bottleneck
        # b = self.bottleneck(self.pool(e2))   # Pool again, then double conv at the bottom of the U

        # Decoder
        #TODO:add a third decoder layer
        d3 = None  # TODO
        d3 = torch.cat([d3, e3], dim=1)
        d3 = None  # TODO

        d2 = self.up2(d3)                   # Upsample bottleneck features
        #d2 = self.up2(b)                   # Upsample bottleneck features
        d2 = torch.cat([d2, e2], dim=1)    # Concatenate skip connection from encoder (same spatial size)
        d2 = self.dec2(d2)                 # Process concatenated features with double conv

        d1 = self.up1(d2)                   # Upsample
        d1 = torch.cat([d1, e1], dim=1)     # Concatenate skip connection from first encoder
        d1 = self.dec1(d1)                  # Process with double conv

        # Output
        return self.out(d1)                 # Final 1x1 convolution to produce output

## Train the three layer U-Net (do not change this code)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

    train_losses = []
    test_losses = []

    for epoch in range(200):
        # ---- Training phase ----
        train_epoch_loss = 0
        model.train()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

            train_epoch_loss += loss.item()

        train_epoch_loss /= len(train_loader)
        train_losses.append(train_epoch_loss)
        print(f"Epoch {epoch+1:02d} | Loss: {train_epoch_loss:.4f}")

        # ---- Testing phase ----
        test_epoch_loss = 0
        model.eval()

        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                loss = criterion(model(x), y)
                test_epoch_loss += loss.item()

        test_epoch_loss /= len(test_loader)
        test_losses.append(test_epoch_loss)
except Exception as e:
    raise NotImplementedError("Something is missing above. Complete all #TODO sections before training.") from e


plt.plot(train_losses, marker="o", label="Train Loss")
plt.plot(test_losses, marker="o", label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")
plt.title("Training loss")
plt.legend()
plt.grid(True)
plt.show()

## Prediction on a new image with the optimised U-Net

In [ ]:
model.eval()
test_img, test_mask = generate_sample()

x = torch.tensor(test_img).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    pred = torch.sigmoid(model(x))[0, 0].cpu()

display_results(test_img, pred, test_mask)


# Advanced Challenge: Noisy and Ambiguous Images

The first part of the exercise used very simple images: one circle and one rectangle.

Now we make the task more realistic. The new images can contain:

- multiple circles
- multiple rectangles
- overlapping shapes
- different brightness levels
- random noise
- rectangles partially overlapping circles

The goal is still the same: **segment the rectangles**.

This lets us ask a more interesting question:

> **Does a U-Net trained on simple images generalise when the input changes?**

We will also use **IoU** and **Dice score** in addition to the training loss.

## Creating a harder dataset

For the advanced dataset, the image is generated with several distractors and target objects.

The mask contains **only the rectangles**. Circles are distractors, even when they overlap with a rectangle.

We generate more examples than before because a more varied problem needs more training data.

You do not need to touch the code below. Just run! The idea is that the code creates a much larger dataset with more difficult samples.

In [ ]:
# Advanced image generator

def generate_advanced_sample(
    image_size=IMAGE_SIZE,
    min_circles=2,
    max_circles=5,
    min_rectangles=1,
    max_rectangles=3,
    noise_std_range=(0.05, 0.15),
):
    image = np.zeros((image_size, image_size), dtype=np.float32)
    mask = np.zeros((image_size, image_size), dtype=np.float32)

    # Add multiple circles (distractors)
    n_circles = np.random.randint(min_circles, max_circles + 1)

    for _ in range(n_circles):
        r = np.random.randint(8, 21)
        cy, cx = np.random.randint(r, image_size - r, size=2)

        rr, cc = disk((cy, cx), r)

        # Circle brightness overlaps with some rectangle brightnesses,
        # making the segmentation task less trivial.
        brightness = np.random.uniform(0.20, 0.70)
        image[rr, cc] = np.maximum(image[rr, cc], brightness)

    # Add multiple rectangles (targets)
    n_rectangles = np.random.randint(min_rectangles, max_rectangles + 1)

    for _ in range(n_rectangles):
        h = np.random.randint(15, 36)
        w = np.random.randint(15, 36)

        ry = np.random.randint(0, image_size - h)
        rx = np.random.randint(0, image_size - w)

        rr, cc = rectangle(start=(ry, rx), extent=(h, w))

        # Rectangle brightness varies from image to image.
        brightness = np.random.uniform(0.40, 1.00)
        image[rr, cc] = np.maximum(image[rr, cc], brightness)

        # The rectangle is always part of the target mask.
        mask[rr, cc] = 1.0

    # Add random Gaussian noise to the entire image.
    noise_std = np.random.uniform(*noise_std_range)
    noise = np.random.normal(0, noise_std, image.shape).astype(np.float32)
    image = np.clip(image + noise, 0.0, 1.0)

    return image, mask


In [ ]:
# A fixed dataset: samples are generated once so evaluation is stable.
from torch.utils.data import Dataset
class AdvancedShapesDataset(Dataset):
    def __init__(self, n_samples, **generator_kwargs):
        self.images = []
        self.masks = []

        for _ in range(n_samples):
            img, mask = generate_advanced_sample(**generator_kwargs)
            self.images.append(img)
            self.masks.append(mask)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.images[idx], dtype=torch.float32).unsqueeze(0),
            torch.tensor(self.masks[idx], dtype=torch.float32).unsqueeze(0)
        )


In [ ]:
# We now use substantially more examples than the original 10-image dataset.
from skimage.draw import rectangle, disk
advanced_train_dataset = AdvancedShapesDataset(
    300,
    noise_std_range=(0.05, 0.15)
)

advanced_test_dataset = AdvancedShapesDataset(
    100,
    noise_std_range=(0.05, 0.15)
)

print("Training examples:", len(advanced_train_dataset))
print("Test examples:", len(advanced_test_dataset))


In [ ]:
# Visualise the harder training examples

def show_advanced_samples(dataset, n=6):
    fig, axes = plt.subplots(3, n, figsize=(3*n, 8))

    for i in range(n):
        img, mask = dataset[i]

        axes[0, i].imshow(img[0], cmap="gray", vmin=0, vmax=1)
        axes[0, i].set_title("Input")
        axes[0, i].axis("off")

        axes[1, i].imshow(mask[0], cmap="gray", vmin=0, vmax=1)
        axes[1, i].set_title("Rectangle mask")
        axes[1, i].axis("off")

        axes[2, i].imshow(img[0], cmap="gray", vmin=0, vmax=1)
        axes[2, i].imshow(mask[0], cmap="Reds", alpha=0.35)
        axes[2, i].set_title("Overlay")
        axes[2, i].axis("off")

    plt.tight_layout()
    plt.show()

show_advanced_samples(advanced_train_dataset)


## First test: does the simple model generalise?

The model from the first part was trained on clean, simple images.

We now give that model the harder images **without retraining it**.

This is a direct test of generalisation: the input distribution has changed.

### Evaluating the Segmentation

The `segmentation_metrics()` function measures how closely the U-Net's predicted mask matches the ground-truth rectangle mask. It calculates two important segmentation metrics: **Intersection over Union (IoU)** and **Dice score**. Both measure the overlap between the predicted and correct regions. IoU is calculated as the intersection of the predicted and true regions divided by their union, while Dice gives twice the intersection divided by the total size of the two regions. Both metrics range from 0 to 1, where **1 means a perfect segmentation and 0 means no overlap**. Therefore, higher IoU and Dice scores indicate better segmentation.

The `evaluate_model()` function applies these measurements to the entire dataset. It runs the U-Net in evaluation mode, calculates the BCE loss, IoU, and Dice score for each batch, and then returns their average values. This allows us to evaluate the model using both its **loss** and its **segmentation quality**.


In [ ]:

def segmentation_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    pred_area = preds.sum(dim=(1, 2, 3))
    target_area = targets.sum(dim=(1, 2, 3))

    union = pred_area + target_area - intersection

    iou = (intersection + 1e-8) / (union + 1e-8)
    dice = (2 * intersection + 1e-8) / (pred_area + target_area + 1e-8)

    return iou.mean().item(), dice.mean().item()


def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    n_batches = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            #TODO: get the model logits
            logits = None  # TODO
            #TODO: calculate the BCE loss
            loss = criterion(None, None)  # TODO
            #TODO: complete the following line
            iou, dice = None, None  # TODO

            total_loss += loss.item()
            total_iou += iou
            total_dice += dice
            n_batches += 1

    return (
        total_loss / n_batches,
        total_iou / n_batches,
        total_dice / n_batches,
    )


In [ ]:
advanced_test_loader = DataLoader(
    advanced_test_dataset,
    batch_size=16,
    shuffle=False
)

old_model_loss, old_model_iou, old_model_dice = evaluate_model(
    model,
    advanced_test_loader,
    criterion
)

print(f"Simple-model loss on harder images: {old_model_loss:.4f}")
print(f"Simple-model IoU:                    {old_model_iou:.3f}")
print(f"Simple-model Dice:                   {old_model_dice:.3f}")


The simple U-Net was trained on clean images, so it has never seen noise, several shapes in one image or dim rectangles.

Its **IoU and Dice** on the harder images may still look reasonable. Now look at the **BCE loss**: it is much higher than on the simple data.

How can the overlap scores look fine while the loss is high?

- IoU and Dice are computed **per image** on a hard 0/1 mask (threshold 0.5) and then averaged. If the model segments one bright rectangle perfectly and completely misses a dim one, that image still gets a fairly good score.
- BCE checks **every pixel** and heavily penalises confident wrong answers, so every missed rectangle pushes the loss up.

Look at the predictions below. Which rectangles does the simple model find, and which does it miss?

In [ ]:
# Visualise how the original simple-data model performs on the harder images.

def show_predictions(model, dataset, n=4):
    model.eval()

    fig, axes = plt.subplots(3, n, figsize=(3*n, 8))

    with torch.no_grad():
        for i in range(n):
            img, mask = dataset[i]
            x = img.unsqueeze(0).to(device)

            pred = torch.sigmoid(model(x))[0, 0].cpu()

            axes[0, i].imshow(img[0], cmap="gray", vmin=0, vmax=1)
            axes[0, i].set_title("Input")
            axes[0, i].axis("off")

            axes[1, i].imshow(pred > 0.5, cmap="gray", vmin=0, vmax=1)
            axes[1, i].set_title("Prediction")
            axes[1, i].axis("off")

            axes[2, i].imshow(mask[0], cmap="gray", vmin=0, vmax=1)
            axes[2, i].set_title("Ground truth")
            axes[2, i].axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, advanced_test_dataset)


## Train a U-Net on the harder problem

Now we give the model examples that contain the same kinds of variation that it will encounter at test time.

We use 300 training images instead of the original 10.

The architecture below is the **three-level U-Net** from the previous part of the exercise. The interesting change is the **data**, not just the architecture.

In [ ]:
# Reuse the three-level U-Net architecture from the previous section.
# If you changed your architecture in the TODO section, this creates a clean
# three-level version for the advanced experiment.

class AdvancedUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(1, 8)
        self.enc2 = DoubleConv(8, 16)
        self.enc3 = DoubleConv(16, 32)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(32, 64)

        self.up3 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec3 = DoubleConv(64, 32)

        self.up2 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.dec2 = DoubleConv(32, 16)

        self.up1 = nn.ConvTranspose2d(16, 8, 2, 2)
        self.dec1 = DoubleConv(16, 8)

        self.out = nn.Conv2d(8, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


In [ ]:
advanced_model = AdvancedUNet().to(device)

advanced_optimizer = torch.optim.Adam(
    advanced_model.parameters(),
    lr=1e-3
)

advanced_criterion = nn.BCEWithLogitsLoss()

advanced_train_loader = DataLoader(
    advanced_train_dataset,
    batch_size=16,
    shuffle=True
)

advanced_test_loader = DataLoader(
    advanced_test_dataset,
    batch_size=16,
    shuffle=False
)

advanced_train_losses = []
advanced_test_losses = []

try:
    for epoch in range(15):
        # ---- Training ----
        advanced_model.train()
        train_loss = 0.0

        for x, y in advanced_train_loader:
            x, y = x.to(device), y.to(device)

            advanced_optimizer.zero_grad()

            logits = advanced_model(x)
            loss = advanced_criterion(logits, y)

            loss.backward()
            advanced_optimizer.step()

            train_loss += loss.item()

        train_loss /= len(advanced_train_loader)
        advanced_train_losses.append(train_loss)

        # ---- Evaluation ----
        test_loss, test_iou, test_dice = evaluate_model(
            advanced_model,
            advanced_test_loader,
            advanced_criterion
        )

        advanced_test_losses.append(test_loss)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"IoU: {test_iou:.3f} | "
            f"Dice: {test_dice:.3f}"
        )
except Exception as e:
    raise NotImplementedError("Something is missing above. Complete all #TODO sections before training.") from e


Compare these numbers with the simple U-Net's results from before.

The IoU and Dice scores may only be somewhat higher, but the **test loss is many times lower**. Remember that IoU and Dice are averaged per image, so a model that only finds the easy rectangles can still score well. The loss shows the real difference: the model trained on the harder data is correct on far more pixels, and confident about it.

In practice, it no longer relies on brightness alone. It also finds the **dim rectangles** that the simple model missed, and it mostly ignores the noise and the circles. You will see this on the even harder challenge set below.

This is why we continue with this model: **one metric on its own can hide important differences**. Always look at the loss, the overlap scores and the predictions together.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(advanced_train_losses, label="Train Loss")
plt.plot(advanced_test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.yscale("log")
plt.title("Advanced U-Net training")
plt.legend()
plt.grid(True)
plt.show()

final_loss, final_iou, final_dice = evaluate_model(
    advanced_model,
    advanced_test_loader,
    advanced_criterion
)

print(f"Final test loss: {final_loss:.4f}")
print(f"Final test IoU:  {final_iou:.3f}")
print(f"Final test Dice: {final_dice:.3f}")


## Test on an even harder distribution

To test generalisation properly, we now create a **new challenge set**.

The model was not trained on these exact images. The challenge set contains:

- more shapes
- stronger noise
- more overlap
- more variation in brightness

This asks:

> **Does the model learn the concept of a rectangle, or does it mainly learn the training distribution?**

In [ ]:
challenge_dataset = AdvancedShapesDataset(
    100,
    min_circles=3,
    max_circles=6,
    min_rectangles=1,
    max_rectangles=4,
    noise_std_range=(0.15, 0.25)
)

challenge_loader = DataLoader(
    challenge_dataset,
    batch_size=16,
    shuffle=False
)

challenge_loss, challenge_iou, challenge_dice = evaluate_model(
    advanced_model,
    challenge_loader,
    advanced_criterion
)

print(f"Challenge loss: {challenge_loss:.4f}")
print(f"Challenge IoU:  {challenge_iou:.3f}")
print(f"Challenge Dice: {challenge_dice:.3f}")

# Evaluate the simple-data model on the same challenge set for comparison
simple_challenge_loss, simple_challenge_iou, simple_challenge_dice = evaluate_model(
    model,
    challenge_loader,
    criterion
)

print(f"Simple-model challenge loss: {simple_challenge_loss:.4f}")
print(f"Simple-model challenge IoU:  {simple_challenge_iou:.3f}")
print(f"Simple-model challenge Dice: {simple_challenge_dice:.3f}")


In [ ]:
show_predictions(advanced_model, challenge_dataset)


## Compare the two models

We can now compare:

1. **Simple-data U-Net** — trained on clean images and then tested on the harder distribution.
2. **Advanced-data U-Net** — trained on noisy and ambiguous images.
3. **Challenge set** — contains a further change in the input distribution.

Record the IoU and Dice scores and discuss:

- Does training on more varied examples help?
- How much does performance change when the test images become harder?
- Is a lower BCE loss always accompanied by a better segmentation?
- What kinds of images are still difficult for the model?
- Would you improve the architecture, the data, or the loss function next?

In [ ]:
# Summary table

results = {
    "Simple U-Net on advanced test": (
        old_model_loss,
        old_model_iou,
        old_model_dice
    ),
    "Advanced U-Net on advanced test": (
        final_loss,
        final_iou,
        final_dice
    ),
    "Simple U-Net on challenge set": (
        simple_challenge_loss,
        simple_challenge_iou,
        simple_challenge_dice
    ),
    "Advanced U-Net on challenge set": (
        challenge_loss,
        challenge_iou,
        challenge_dice
    ),
}

for name, (bce, iou, dice) in results.items():
    print(f"{name:40s} | BCE: {bce:.4f} | IoU: {iou:.3f} | Dice: {dice:.3f}")


# Final Challenge

You have now seen that making a segmentation problem harder is not only about making the neural network deeper.

The main challenge is **generalisation**.

Once you have completed the main exercise, try changing **one parameter at a time** and observe how it affects the model's performance. (Treat this as a bonus exercise)

Experiment with:

- **Noise level:** Try different `noise_std_range` values.
- **Number of training examples:** Increase or decrease the dataset size.
- **Number of epochs:** Train the model for longer or shorter.


### Key takeaway

> A good segmentation model should not only perform well on the images it was trained on. It should continue to produce useful masks when the input changes.